#  Scientfic Computing

Material of the course Scientific Computing, Instituto Tecnológico Metropolitano.

Author: Sebastián Roldán Vasco

Created on May 02, 2024.


---


# Data analysis

The famous [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set) includes three species of Iris flowers: Setosa, Versicolor, and Virginica. This dataset is often used for learning purposes in machine learning and data analysis.


## Load the Dataset

We'll start by loading the Iris dataset into our analysis environment. This dataset is readily available in many machine learning libraries like `scikit-learn`, as well as in visualization libraries as `seaborn`.



In [ ]:
# Import and install  the necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, shapiro, kruskal
!pip install skrebate
!pip install mrmr-selection

from skrebate import ReliefF
from mrmr import mrmr_classif

In [ ]:
iris = load_iris()
print(iris.DESCR)

In [ ]:
print(type(iris))

Observe that `iris` is an object of the classs `sklearn`. As we saw in the previous lecture, `pandas` is a powerful tool for data analysis, so let's create a `pandas` `DataFrame`:


In [ ]:
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

Every species has an assigned label, as follows:

* 0 represents the Setosa species
* 1 represents the Versicolor species
* 2 represents the Virginica species

In [ ]:
iris.target

In [ ]:
# Add the target variable to the DataFrame
iris_df['species'] = iris.target

print(iris_df.head())

## Visual exploratory data analysis

Exploring the Iris dataset involves understanding its structure, features, and the distribution of data through summary statistics and visualizations. There are several ways to do so. Next, some types of explorations are performed.

**Summary of statistics**

* Calculate summary statistics such as mean, median, standard deviation, minimum,
and maximum for each feature.

* Use the `describe()` method in `pandas` DataFrame to get a summary of numerical features.

In [ ]:
print("Summary of statistics:")
print(iris_df.describe())

**Data visualization**
* Create histograms to visualize the distribution of each feature.
* Generate box plots to identify any outliers in the data.
* Plot pairwise feature scatter plots to explore relationships between different features.
* Use violin plots or swarm plots to compare feature distributions across different species.

In [ ]:
# Histograms of each feature
iris_df.hist(figsize=(10, 8))
plt.tight_layout()
plt.show()

Notwhitstanding, the previous figures do not show the distribution per class, to figure out if there is class separability:

In [ ]:
for i, feature in enumerate(iris.feature_names):
    plt.subplot(2, 2, i + 1)
    sns.histplot(data=iris_df, x=feature, hue='species', kde=False, palette='Set2', alpha=0.7)
    plt.title(f'Histogram of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

Even though previous plots illustrate the data distribution, it's hard to analyze the class separability. Instead, we can use density plots by 1) changing the `kde` option to `True`, or by plotting the density plot directly:

In [ ]:
for i, feature in enumerate(iris.feature_names):
    plt.subplot(2, 2, i + 1)
    sns.histplot(data=iris_df, x=feature, hue='species', kde=True, palette='Set2', alpha=0.7)
    plt.title(f'Histogram of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
for i, feature in enumerate(iris.feature_names):
    plt.subplot(2, 2, i + 1)
    sns.kdeplot(data=iris_df, x=feature, hue='species', fill=True, alpha=0.5, linewidth=0.5)
    plt.title(f'Density Plot of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Density')

plt.tight_layout()
plt.show()

What can we conclude about the class separability?

In [ ]:
# Box plot of each feature
plt.figure(figsize=(10, 8))
sns.boxplot(data=iris_df.drop('species', axis=1))
plt.title("Box plot of Features")
plt.show()

Again, previous figures show us the general panorama, i.e., we can analyze the lenght or width of sepal and then compare with the petal dimensions. However, it would be more interesting to find differences between species:

In [ ]:
for i, feature in enumerate(iris.feature_names):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(x='species', y=feature, data=iris_df) #palette='Set2')
    plt.title(f'Boxplot of {feature} by Species')
    plt.xlabel('Species')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

And now, we can combine the characteristics of box plots and a kernel density plots by violin plots. We can use them to visualize the distribution of each feature for each species of Iris flower.

In [ ]:
for i, feature in enumerate(iris.feature_names):
    plt.subplot(2, 2, i + 1)
    sns.violinplot(x='species', y=feature, data=iris_df, palette='Set2')
    plt.title(f'Violin Plot of {feature}')
    plt.xlabel('Species')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

This is another good option to plot the differences between classes according to the feature.

**Is there any relationship between features?**

One graphical way to find how correlated are two features is the plotting of pairwise scatter plots. It also serves to figure out statistical dependencies.

In [ ]:
sns.pairplot(iris_df, hue='species', palette='Set2', markers=["o", "s", "D"], diag_kind='kde')
plt.suptitle("Pairplot of Iris Dataset by Species", y=1.02)
plt.show()

## Class separability

There are many ways to determine how "separated" are two -or more- different classes. In the current example, classes are flower species. Let use different strategies and compare the results.

**Hypothesis tests**

Let's compute a hypothesis test for the Iris dataset petals. First, we must check the data normality:

In [ ]:
setosa_petal_length = iris_df[iris_df['species'] == 0]['petal length (cm)']
versicolor_petal_length = iris_df[iris_df['species'] == 1]['petal length (cm)']
virginica_petal_length = iris_df[iris_df['species'] == 2]['petal length (cm)']

# Perform the Shapiro-Wilk test for normality
shapiro_setosa = shapiro(setosa_petal_length)
shapiro_versicolor = shapiro(versicolor_petal_length)
shapiro_virginica = shapiro(virginica_petal_length)

print("Shapiro-Wilk Test for Setosa: ", shapiro_setosa)
print("Shapiro-Wilk Test for Versicolor: ", shapiro_versicolor)
print("Shapiro-Wilk Test for Virginica: ", shapiro_virginica)

"Small" p-values suggests that the weights likely do not come from a normal distribution. However, in this case, p-value$>0.05$, suggesting data were drawn from a normal distribution.

Thus, bearing in mind the recommendation made by Marusteri & Bacarea (2010), we can perform an ANOVA test:


In [ ]:
h_stat, p_value = f_oneway(setosa_petal_length, versicolor_petal_length, virginica_petal_length)

print(f"H-statistic: {h_stat}")
print(f"P-value: {p_value}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("We reject the null hypothesis. There is a significant difference in the variance of petal lengths among the three species.")
else:
    print("We fail to reject the null hypothesis. There is no significant difference in the variance of petal lengths among the three species.")

**Task**: Modify the previous code to perform the proper statistic according to the normality test.

**Fisher Discriminant Ratio**

The FDR can be represented as follows:

$
\text{FDR} = \frac{{\text{Tr}(\mathbf{W}^{-1} \mathbf{B})}}{{\text{Tr}(\mathbf{W}^{-1} \mathbf{W})}}
$

Where:
- $\mathbf{W}$ is the within-class scatter matrix.
- $\mathbf{B}$ is the between-class scatter matrix.
- $\text{Tr}(\cdot)$ represents the trace of a matrix, which is the sum of its diagonal elements.


The means used in FDR calculation typically represent the means of each feature for each class (species).

For $\mathbf{W}$:


$
\text{Mean of feature } j \text{ in class } i = \frac{{\sum_{k=1}^{n_i} x_{ijk}}}{{n_i}}
$

Where:
- $x_{ijk}$ is the $j$-th feature value of the $k$-th sample in class $i$.
- $n_i$ is the number of samples in class $i$.

For $\mathbf{B}$:


$
\text{Overall mean of feature } j = \frac{{\sum_{i=1}^{c} n_i \times \text{Mean of feature } j \text{ in class } i}}{{\sum_{i=1}^{c} n_i}}
$

Where:

- $c$ is the number of classes (species) in the dataset.

These means are used to compute the within-class and between-class scatter matrices.


In [ ]:
# Calculate means for each feature by species
feature_means_by_species = iris_df.groupby('species').mean()

print('Mean of feature j in class i:\n')
print(feature_means_by_species)

# Calculate overall mean for each feature
overall_mean = iris_df.drop('species', axis=1).mean()


print('\nOverall mean of feature j:\n')
print(overall_mean)



# Calculate within-class scatter matrix
within_class_scatter = np.zeros((len(iris.feature_names), len(iris.feature_names)))
for species in range(3):
    species_df = iris_df[iris_df['species'] == species].drop('species', axis=1)
    species_means = feature_means_by_species.iloc[species]
    within_class_scatter += np.dot((species_df - species_means).T, (species_df - species_means))

# Calculate between-class scatter matrix
between_class_scatter = np.zeros((len(iris.feature_names), len(iris.feature_names)))
for species, species_means in feature_means_by_species.iterrows():
    n = len(iris_df[iris_df['species'] == species])
    between_class_scatter += n * np.outer((species_means - overall_mean), (species_means - overall_mean))

# Calculate Fisher's discriminant ratio for each feature
fisher_discriminant_ratio = np.diag(np.dot(np.linalg.inv(within_class_scatter), between_class_scatter))

# Print Fisher's discriminant ratio for each feature
print('\nFDR:\n')
for i, feature in enumerate(iris.feature_names):
    print(f'{feature}: {fisher_discriminant_ratio[i]}')

Higher values of FDR suggest that the feature is more effective at separating the classes (species) in the dataset:

- Petal length and petal width are highly effective features for discriminating between the Iris species, with significantly **larger between-class scatter** compared to within-class scatter.
- Sepal width shows moderate effectiveness in discriminating between the species.
- Sepal length appears to be less effective compared to the other features in discriminating between the species.


**AUC**

Even useful, the AUC works well for bi-class problems. Thus, we need to reformulate the problem as a binary classification task. One way to do that:

1. Select one feature at a time.
2. Treat the selected feature as the predictor variable and the target variable as a binary variable indicating whether a sample belongs to a certain class or not.
3. Calculate the AUC for each feature individually.

In [ ]:
X = iris.data
y = iris.target

auc_scores = []
for feature_idx in range(X.shape[1]):

    feature_values = X[:, feature_idx]

    for class_idx in np.unique(y):
        # Treat the samples of this class as positive and others as negative
        target = np.where(y == class_idx, 1, 0)

        auc_score = roc_auc_score(target, feature_values)
        auc_scores.append(auc_score)

for feature_idx, auc_score in enumerate(auc_scores):
    print(f"AUC for feature {feature_idx+1}: {auc_score}")

How is it possible, if we have only four features? Modify the code to show the name of the feature and the bi-class scenario.

**ANOVA**

We can apply Analysis of Variance to the Iris dataset to evaluate the significance of each feature with respect to the target variable (species)

In [ ]:
for feature_idx in range(X.shape[1]):
    feature_values = [X[y == i, feature_idx] for i in range(len(iris.target_names))]
    f_statistic, p_value = f_oneway(*feature_values)
    print(f"Feature {feature_idx}: F-statistic={f_statistic}, p-value={p_value}")


- Feature 0 (Sepal Length):
  - The high ANOVA score and very low p-value indicate that the sepal length feature significantly contributes to the differentiation between the iris species. There is strong evidence to reject the null hypothesis, suggesting that the mean sepal lengths are significantly different across the species.

- Feature 1 (Sepal Width):
  - Similarly, the high ANOVA score and very low p-value indicate that the sepal width feature significantly contributes to the differentiation between the iris species. Again, there is strong evidence to reject the null hypothesis.

- Feature 2 (Petal Length):
  - The extremely high ANOVA score and very low p-value suggest that the petal length feature is highly informative for distinguishing between the iris species. There is *overwhelming* evidence to reject the null hypothesis.

- Feature 3 (Petal Width):
  - Similar to petal length, the extremely high ANOVA score and very low p-value indicate that the petal width feature is highly informative for distinguishing between the iris species. There is strong evidence to reject the null hypothesis.

**Relief-F**


Relief-F (Relief Feature Selection) is a feature selection algorithm that assesses the importance of features by considering their relevance with the target variable and the redundancy between features. It is particularly useful for classification tasks and is based on the concept of *nearest neighbors*.

In [ ]:
# Initialize ReliefF feature selector
selector = ReliefF(n_neighbors=5, n_features_to_select=4)

selector.fit(X, y)

selected_feature_indices = selector.top_features_

selected_feature_names = [iris.feature_names[i] for i in selected_feature_indices]

print("Selected features in descending order:", selected_feature_names)


**mRMR**


Minimum Redundancy Maximum Relevance (mRMR) is a feature selection algorithm that aims to select a subset of features that have high relevance with the target variable (maximum relevance) while also minimizing redundancy between the selected features (minimum redundancy).

In [ ]:
df = pd.DataFrame(X, columns=iris.feature_names)

selected = mrmr_classif(X=df, y=y, K=2) # Select top 2 features

print("Selected features:", selected)
